In [ ]:
from data import load_data
import matplotlib.pyplot as plt
import numpy as np
from models.layer.initializer import Initializer
from rich.table import Table
from rich.console import Console
from models.layer.activation import Relu, Softmax
from models.layer import Layer
from scripts.train import create_and_train_model_from_config
from config.config import Config
from typing import Any
from utils import plot_all, plot_confusion_matrix
import shutil
import yaml

In [ ]:
data = load_data()

In [ ]:
model = create_and_train_model_from_config(
    cfg=Config.read_yaml(),
)

In [ ]:
print(
    "Train acc", model.evaluate(X=data.X_train, y=data.y_train)
)  # to verify I didn't forget something
print(
    "Val acc", model.evaluate(X=data.X_val, y=data.y_val)
)  # to verify I didn't forget something

In [ ]:
print(
    "Test acc", model.evaluate(X=data.X_test, y=data.y_test)
)  # to verify I didn't forget something

In [ ]:
y_preds = model.predict(data.X_test)
y_actual = np.argmax(data.y_test, axis=1)
plt.figure(figsize=(10, 10))
plot_confusion_matrix(y_true=y_actual, y_pred=y_preds, normalize=False)
plt.savefig(f"../../report/images/q2_test_confusion.png")
plt.show()

In [ ]:
table = Table()
table.add_column("class")
table.add_column("precision")
table.add_column("recall")
table.add_column("f1")
precisions = []
recalls = []
for i in range(10):
    # recall
    mask = y_actual == i
    recall = np.count_nonzero(y_preds[mask] == y_actual[mask]) / len(y_actual[mask])
    # precision
    mask = y_preds == i
    precision = np.count_nonzero(y_preds[mask] == y_actual[mask]) / len(y_preds[mask])
    # f1
    f1 = 2 * precision * recall / (precision + recall)
    precisions.append((precision, i))
    recalls.append((recall, i))
    table.add_row(
        str(i), str(round(precision, 6)), str(round(recall, 6)), str(round(f1, 6))
    )
console = Console()
console.log(table)
precisions.sort()
recalls.sort()
print(precisions)
print("Precision", [i[1] for i in precisions])
print("Recall", [i[1] for i in recalls])

In [ ]:
mask = ~(y_preds == y_actual)
X_miss = data.X_test[mask]
y_miss = y_actual[mask]
pred_miss = y_preds[mask]
print(np.count_nonzero(mask))
print(np.unique_counts(y_miss))
print(np.unique_counts(pred_miss))

In [ ]:
def plot_missed(X, y, pred, nrows: int, ncols: int):
    for i in range(nrows):
        for j in range(ncols):
            ind = i * ncols + j
            plt.subplot(nrows, ncols, ind + 1)
            plt.axis("off")
            plt.title(f"pred = {pred[ind]}, actual = {y[ind]}")
            plt.imshow(
                X[ind].reshape(28, 28),
                cmap="gray",
            )


plt.figure(figsize=(15, 6))
plot_missed(X_miss, y_miss, pred_miss, 3, 5)
plt.savefig(f"../../report/images/q2_missed_pics.png")

In [ ]:
mask = y_miss == 9
plt.figure(figsize=(13, 3))
plot_missed(X_miss[mask], y_miss[mask], pred_miss[mask], 1, 5)
plt.savefig(f"../../report/images/q2_missed_pics_9.png")